In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Uso do workload

<span id="usage"></span>
Uso é uma medida do tempo em que o QPU está bloqueado para o seu workload, e é calculado de forma diferente dependendo do modo de execução que você está usando.

* O uso de sessão é o tempo de relógio de parede em que a sessão está ativa. Veja [Duração da sessão](/guides/run-jobs-session#session-length) para mais informações sobre a transição de status da sessão.
* O uso de batch é a soma do tempo quântico (tempo gasto pelo complexo QPU para processar seu job) de todos os jobs no batch.
* O uso de um único job é o tempo quântico que o job usa no processamento.

Observe que jobs com falha ou cancelados contam para o seu uso em certas circunstâncias — veja a seção [Jobs com falha e cancelados](#failed-job) para detalhes.

Para usuários de planos pagos, o uso determina quanto o workload custa. Veja [Gerenciar custo](/guides/manage-cost) para detalhes.

<span id="failed-job"></span>
## Uso para jobs com falha e cancelados
Quando um job falha ou é cancelado, o uso reportado é o seguinte:

* Modo job ou batch: O uso reportado é o tempo em que o QPU estava bloqueado para executar seu workload até o momento em que falhou ou foi cancelado. Portanto, se a falha ou cancelamento ocorreu antes do bloqueio, o uso reportado é zero. Caso contrário, o uso reportado do workload é a quantidade de uso antes de ele falhar ou ser cancelado. Assim, alguns jobs com falha não aparecem no seu uso reportado e outros sim.

* Modo sessão: O uso reportado é o tempo de relógio de parede em que a sessão está ativa, independentemente do número de jobs que falham ou são cancelados.

<span id="view-usage"></span>
## Consultar o uso real de um workload
Após a conclusão de um workload, há várias formas de visualizar seu uso real:

- Execute [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) ou [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) no `qiskit-ibm-runtime` 0.30 ou posterior. Se estiver usando uma versão mais antiga do `qiskit-ibm-runtime` (>= 0.23 e < 0.30), o uso ainda pode ser encontrado em `session.details()["usage_time"]` e `batch.details()["usage_time"]`.
- Use [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) para ver o uso de um batch ou sessão específico.
- Use [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) para ver o uso de um único job.

<span id="instance-usage"></span>
## Visualizar o uso da instância
Você pode visualizar o uso de uma instância na página [Instâncias](https://quantum.cloud.ibm.com/instances) ou, para quem tiver a autoridade adequada, na página [Analytics](https://quantum.cloud.ibm.com/analytics). Observe que as páginas podem mostrar números de uso diferentes porque calculam o uso de formas diferentes.

A página Instâncias exibe o uso em tempo real dos últimos 28 dias (contínuo), até o momento atual do dia corrente. O uso da página Analytics é recalculado a cada hora e inclui os últimos 28 dias completos; ou seja, exibe o uso de 00:00 de 28 dias atrás até hoje, no topo da hora.

## Estimar o uso antes de enviar um job
Embora obter uma estimativa local precisa seja complicado pelas operações extras realizadas para supressão e mitigação de erros, você pode usar esta fórmula de referência para obter uma aproximação do uso estimado:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` é uma sobrecarga de aproximadamente 2s por sub-job. Isso inclui operações como carregar o payload na eletrônica de controle. Seu job primitive pode ser dividido em múltiplos sub-jobs se for grande demais para o motor de execução processar de uma vez.
- `rep_delay` é uma opção [personalizável pelo usuário](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), e o padrão é dado por `backend.default_rep_delay`, que é 250 microssegundos na maioria dos backends IBM Quantum. Observe que diminuir o `rep_delay` reduz o tempo total de execução no QPU, mas ao custo de uma taxa de erro de preparação de estado aumentada; veja o guia [Execução com taxa de repetição dinâmica](/guides/repetition-rate-execution) para mais informações.
- `<circuit length>` é o comprimento total das instruções. Cada instrução leva um tempo diferente no QPU, portanto o comprimento total varia de circuit para circuit. Uma medição, por exemplo, pode levar 56 vezes mais tempo que uma gate `x`. `backend.target[<instruction>][<qubit>].duration` pode ser usado para encontrar a duração exata de cada instrução. Um comprimento de circuit típico fica entre 50-100 microssegundos. Se você estiver usando técnicas de supressão ou mitigação de erros com as primitives, instruções extras podem ser inseridas no seu circuit, o que aumentaria o comprimento total do circuit.
    > **Note:** A [opção experimental `scheduler_timing`](/guides/visualize-circuit-timing) retorna o tempo total do circuit, mas este NÃO é o tempo usado para cobrança.
- `<num executions>` é o número total de circuits multiplicado pelo número de shots, onde os circuits são aqueles gerados após os elementos PUB serem transmitidos.
    - Se você estiver usando técnicas de mitigação de erros com as primitives, circuits extras podem ser executados como parte do processo de mitigação, o que aumentaria o número total de execuções. Além disso, técnicas avançadas de mitigação de erros, como PEA e PEC, têm uma sobrecarga muito maior porque exigem a execução de circuits para aprendizado de ruído.
    - O Estimator agrupa observáveis com comutação qubit a qubit, o que reduz o número de execuções.

Se você não estiver usando nenhuma técnica avançada de mitigação de erros ou `rep_delay` personalizado, pode usar `2+0.00035*<num executions>` como uma fórmula rápida.

## Próximos passos
> **Tip:** - Revise estas dicas: [Minimizar o tempo de execução do job](minimize-time).
>     - Defina o [Tempo máximo de execução](max-execution-time).
>     - Aprenda como transpilar localmente na seção [Transpile](./transpile/).
>     - Experimente o guia [Comparar configurações do Transpiler](/guides/circuit-transpilation-settings).